# Celdas nuevas — comparación cruzada estimador ↔ analógico renormalizado

Estas celdas se insertan en `convergencia_renormalizacion.ipynb` **después de la
§4** (renormalización) y reutilizan todo lo que ya existe ahí: `perfiles`,
`renorm`, `renormaliza`, `fila`, `NS`, `NMAX`, `costo`, `runs`, `sweeps`,
`espejo`, `Q_CONO`, `T_CRIT`, `ESTILO`, `MARCA`, `COLOR_MEDIO`.

**Todo lo que sigue compara el estimador contra el analógico renormalizado**
(`eta_ren`, `err_ren`, y el ápice ya corregido por $A$ en `costo`). No aparece el
analógico crudo salvo en la Fig. 1a, que es lo que justifica la renormalización.

Lo que agregan, en orden:

| § | qué faltaba | salida |
|---|---|---|
| 4b | réplicas renormalizadas + interpolación a grilla común | `eta_rep_ren`, `en_grilla` |
| 5b | ¿la diferencia entre modos es ruido o es sesgo? | `tabla_consistencia.csv`, **Fig. 1c** |
| 6b | $D_{\rm rms}$ **contra el otro modo**, no contra sí mismo | `tabla_convergencia.csv` |
| 6c | ajuste $D^2 = a/N + c^2$ con exponente fijo en $-1/2$ | `tabla_ajuste_convergencia.csv` |
| 6d | la figura de convergencia cruzada | **Fig. 3b** |
| 7b | speedup **a igual precisión**, no a igual $N$ | `tabla_igual_precision.csv` |
| 7c | por qué la mezcla cuesta lo que cuesta | costo por historia + params |

In [2]:
import sys
import os
sys.path.append(os.path.abspath("../"))
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable

from utils.loaders import load_sweep
from utils.styles import apply, TEXTWIDTH_IN, DOC_FONTSIZE, COL
from utils.analysis import cbs_profiles, linear, circular, phi_cut, azimuthal_average

In [5]:
# --- rutas -----------------------------------------------------------------
save_path = "/Users/niaggar/Results"

FIGDIR = Path("figs_tesis")
FIGDIR.mkdir(exist_ok=True)

# Nombres de carpeta de las tres campañas gemelas. AJUSTAR si difieren:
# el notebook salta sin quejarse las que no encuentre.
CAMPAIGNS = {
    "Homogeneous": "study_convergence_homogeneous_r55nm__CIRCULAR__beam2500",
    "Mixture":     "study_convergence_mixture__CIRCULAR__beam2500",
    "Bilayer":     "study_convergence_layers__CIRCULAR__beam2500",
}
MEDIA_ORDER = ["Homogeneous", "Mixture", "Bilayer"]

# --- física ----------------------------------------------------------------
N_MEDIUM   = 1.33
WAVELENGTH = 0.514                                   # um
K_MEDIUM   = 2 * np.pi * N_MEDIUM / WAVELENGTH       # 1/um

# Incidencia circular -> canal de helicidad conservada. En base circular
# "cross" = I_+ (el que va a eta(0) = 2 por reciprocidad).
BASIS   = circular
CHANNEL = "cross"
REDUCER = azimuthal_average       # obligatorio: el modo directo no admite
                                  # comparacion columna a columna en phi
SENSOR  = "farfield_cbs"          # detector unico (sin stitching)

# --- estadistica -----------------------------------------------------------
N_REPLICAS  = 5
T_CRIT      = 2.776               # t_{0.975,4}
Q_CONE      = 7.5                 # ventana del cono para las metricas de perfil
Q_TAIL_FRAC = 0.6                 # cola usada para la baseline incoherente
EPS         = 1e-30

# --- estilo ----------------------------------------------------------------
MODE_ORDER = ["estimator", "analog"]
MODE_STYLE = {
    "estimator": dict(color=COL[0], marker="o", ls="-"),
    "analog":    dict(color=COL[2], marker="s", ls="--"),
}
MEDIA_COLOR = {m: COL[i] for i, m in enumerate(MEDIA_ORDER)}
CMAP_N = plt.get_cmap("viridis")

sweeps = {}
for label, folder in CAMPAIGNS.items():
    try:
        sweeps[label] = load_sweep(folder, base_path=Path(save_path))
        print(f"{label:12s}: {len(sweeps[label]):3d} datasets   <- {folder}")
    except Exception as err:
        print(f"{label:12s}: NO DISPONIBLE ({type(err).__name__}: {err})")

MEDIA = [m for m in MEDIA_ORDER if m in sweeps]
print("\ncampanas activas:", MEDIA)

Homogeneous :  70 datasets   <- study_convergence_homogeneous_r55nm__CIRCULAR__beam2500
Mixture     :  70 datasets   <- study_convergence_mixture__CIRCULAR__beam2500
Bilayer     :  70 datasets   <- study_convergence_layers__CIRCULAR__beam2500

campanas activas: ['Homogeneous', 'Mixture', 'Bilayer']


In [6]:
def lstar_anchor(p):
    """l* con el que se construyo la grilla angular (q = k l* theta).

    Se prueban las claves en orden de especificidad; la primera que exista y
    sea > 0 gana. Si ninguna esta, la corrida no es interpretable en q."""
    for name in ("lstar_angle_anchor", "lstar_mix", "lstar_in",
                 "transport_mean_free_path", "lstar_sim"):
        v = p.get(name)
        if v is not None and np.isfinite(float(v)) and float(v) > 0:
            return float(v)
    raise KeyError("sin ancla de l* en params_flat")


def build_index(label, sweep):
    """Una fila por REPLICA."""
    rows = []
    for key in sweep.keys():
        p = sweep[key].params_flat
        mode = p.get("mode")
        if mode not in MODE_ORDER:          # corrida ajena a esta campana
            continue
        n_chunk = int(p.get("n_photons", 0))
        n_reps  = int(p.get("n_replicas", N_REPLICAS))
        rows.append(dict(
            medium  = label,
            key     = key,
            mode    = mode,
            idx     = int(p.get("ladder_index", -1)),
            rep     = int(p.get("replica", -1)),
            n_chunk = n_chunk,
            n_total = int(p.get("n_photons_point_total", n_chunk * n_reps)),
            wall_s  = float(p.get("runtime_s", np.nan)),
            cpu_s   = float(p.get("cpu_time_s", np.nan)),
            hits    = float(p.get("hits", np.nan)),
            lstar   = lstar_anchor(p),
        ))
    return pd.DataFrame(rows)


if not MEDIA:
    raise RuntimeError("ninguna campana encontrada: revisar save_path y CAMPAIGNS")

runs = pd.concat([build_index(m, sweeps[m]) for m in MEDIA], ignore_index=True)
runs = runs.sort_values(["medium", "mode", "n_total", "rep"]).reset_index(drop=True)

print(runs.groupby(["medium", "mode"])
          .agg(points=("n_total", "nunique"),
               runs=("key", "size"),
               N_min=("n_total", "min"),
               N_max=("n_total", "max"),
               wall_h=("wall_s", lambda s: s.sum() / 3600.0))
          .round(2))

                       points  runs    N_min       N_max  wall_h
medium      mode                                                
Bilayer     analog          7    35  1000000  1000000000    4.82
            estimator       7    35     1000      500000    0.75
Homogeneous analog          7    35  1000000  1000000000    1.45
            estimator       7    35     1000      500000    0.23
Mixture     analog          7    35  1000000  1000000000    1.52
            estimator       7    35     1000      500000    2.65


In [7]:
def raw_pair(sweep, key, lstar, time_index=0):
    """(q, coherente, incoherente) de UNA replica, sin normalizar."""
    p = cbs_profiles(sweep[key].processed_cbs(SENSOR),
                     basis=BASIS, time_index=time_index, reduce=REDUCER)
    q = K_MEDIUM * lstar * np.asarray(p.theta, float)
    return (q,
            np.asarray(p.coherent[CHANNEL],   float),
            np.asarray(p.incoherent[CHANNEL], float))


def tail_baseline(q, y, frac=Q_TAIL_FRAC):
    """Fondo incoherente = mediana de la cola. Robusto a pedestales residuales."""
    m = q > frac * q[-1]
    if m.sum() < 5:
        raise ValueError("cola insuficiente para estimar el fondo")
    return np.median(y[m])


def peak_height(q, E, n=3):
    """Apice como mediana de los primeros n bins (inmune a un bin ruidoso)."""
    return float(np.median(E[:n]))


def fwhm_q(q, E, n=3, sustain=2):
    """FWHM en q respecto al fondo incoherente.

    sustain: bins consecutivos bajo el semi-maximo exigidos para aceptar el
    cruce (a N bajo el ruido cruza y vuelve). Devuelve nan si no hay cono."""
    B  = tail_baseline(q, E)
    E0 = peak_height(q, E, n)
    if E0 <= B:
        return np.nan
    half  = B + 0.5 * (E0 - B)
    below = E < half
    for i in range(1, len(E) - sustain + 1):
        if below[i:i + sustain].all():
            break
    else:
        return np.nan
    if below[:i].all():
        return np.nan
    j = i - 1
    while j > 0 and E[j] < half:
        j -= 1
    q_half = q[j] + (half - E[j]) * (q[j + 1] - q[j]) / (E[j + 1] - E[j])
    return 2.0 * q_half


def mirror(q, *ys):
    """Espejo -q..+q (valido: I(th,phi) = I(th,phi+pi))."""
    return (np.r_[-q[::-1], q],) + tuple(np.r_[y[::-1], y] for y in ys)


def eta_point(sweep, keys, lstar, time_index=0):
    """Perfil de UN punto de la escalera.

    Devuelve (q, eta_pooled, semiancho_IC, eta_por_replica)."""
    C = I = q = None
    per_rep = []
    for k in keys:
        q, c, i = raw_pair(sweep, k, lstar, time_index)
        C = c if C is None else C + c
        I = i if I is None else I + i
        e = (c + EPS) / (i + EPS)
        per_rep.append(e - tail_baseline(q, e) + 1.0)

    eta = (C + EPS) / (I + EPS)
    eta = eta - tail_baseline(q, eta) + 1.0

    per_rep = np.vstack(per_rep)
    R = per_rep.shape[0]
    half = T_CRIT * per_rep.std(axis=0, ddof=1) / np.sqrt(R)
    return q, eta, half, per_rep


def build_points(medium):
    """{(modo, N_total): dict con perfil, escalares y costo}."""
    sweep = sweeps[medium]
    sub   = runs[runs.medium == medium]
    out   = {}
    for (mode, n_total), grp in sub.groupby(["mode", "n_total"]):
        grp   = grp.sort_values("rep")
        lstar = float(grp.lstar.iloc[0])
        q, eta, half, per_rep = eta_point(sweep, grp.key.tolist(), lstar)
        R = per_rep.shape[0]

        e0_rep = np.array([peak_height(q, e) for e in per_rep])
        dq_rep = np.array([fwhm_q(q, e)      for e in per_rep])

        out[(mode, int(n_total))] = dict(
            q=q, eta=eta, half=half, per_rep=per_rep, R=R, lstar=lstar,
            eta0   = peak_height(q, eta),
            dq     = fwhm_q(q, eta),
            # sigma del punto COMPLETO = s_replica / sqrt(R)
            s_eta0 = np.nanstd(e0_rep, ddof=1) / np.sqrt(R),
            s_dq   = np.nanstd(dq_rep, ddof=1) / np.sqrt(R),
            T_wall = float(grp.wall_s.sum()),
            T_cpu  = float(grp.cpu_s.sum()),
            hits   = float(grp.hits.sum()),
        )
    return out


POINTS = {m: build_points(m) for m in MEDIA}
LADDER = {(m, mode): sorted(n for (md_, n) in POINTS[m] if md_ == mode)
          for m in MEDIA for mode in MODE_ORDER}
NMAX   = {(m, mode): LADDER[(m, mode)][-1] for m in MEDIA for mode in MODE_ORDER
          if LADDER[(m, mode)]}

for m in MEDIA:
    for mode in MODE_ORDER:
        print(f"{m:12s} {mode:9s}: N = {[f'{n:.0e}' for n in LADDER[(m, mode)]]}")

/Users/niaggar/Developer/luminis-mc/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/niaggar/Developer/luminis-mc/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Homogeneous  estimator: N = ['1e+03', '3e+03', '1e+04', '3e+04', '1e+05', '3e+05', '5e+05']
Homogeneous  analog   : N = ['1e+06', '3e+06', '1e+07', '3e+07', '1e+08', '5e+08', '1e+09']
Mixture      estimator: N = ['1e+03', '3e+03', '1e+04', '3e+04', '1e+05', '3e+05', '5e+05']
Mixture      analog   : N = ['1e+06', '3e+06', '1e+07', '3e+07', '1e+08', '5e+08', '1e+09']
Bilayer      estimator: N = ['1e+03', '3e+03', '1e+04', '3e+04', '1e+05', '3e+05', '5e+05']
Bilayer      analog   : N = ['1e+06', '3e+06', '1e+07', '3e+07', '1e+08', '5e+08', '1e+09']


/Users/niaggar/Developer/luminis-mc/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [8]:
# --- chequeo: la grilla en q debe ser IDENTICA en toda la campana -----------
for m in MEDIA:
    qs = [d["q"] for d in POINTS[m].values()]
    same = all(np.allclose(qs[0], x) for x in qs)
    print(f"{m:12s}: grilla q identica = {same} | dq = {np.diff(qs[0]).mean():.4f} "
          f"| q_max = {qs[0][-1]:.1f} | l*_ancla = "
          f"{ {round(d['lstar'], 4) for d in POINTS[m].values()} }")

Homogeneous : grilla q identica = True | dq = 0.0200 | q_max = 40.0 | l*_ancla = {39.7958}
Mixture     : grilla q identica = True | dq = 0.0200 | q_max = 40.0 | l*_ancla = {20.4125}
Bilayer     : grilla q identica = True | dq = 0.0200 | q_max = 40.0 | l*_ancla = {23.2166}


In [11]:
# =========================================================================
# E -- comparacion a resolucion comun (rebin con peso de angulo solido)
# =========================================================================
def rebin_pair(q, C, I, dq_new):
    """Re-binea los ACUMULADORES (nunca el cociente) a una grilla mas gruesa.

    C e I ya vienen normalizados por angulo solido, asi que el promedio
    correcto pesa por dOmega ~ sin(theta) dtheta dphi ~ q. El q de salida es
    el CENTROIDE pesado por dOmega, que es el angulo al que el modo directo
    realmente reporta -- no el centro geometrico. Para el primer bin la
    diferencia es 4/3."""
    edges = np.arange(0.0, q[-1] + dq_new, dq_new)
    idx   = np.clip(np.digitize(q, edges) - 1, 0, len(edges) - 2)
    w     = np.maximum(q, 1e-12)                      # dOmega
    n     = len(edges) - 1
    W     = np.bincount(idx, weights=w, minlength=n)
    good  = W > 0
    agg   = lambda y: np.bincount(idx, weights=y * w, minlength=n)[good] / W[good]
    return agg(q), agg(C), agg(I)


def eta_point_rebin(medium, mode, dq_new, n=None):
    """Como eta_point pero a resolucion dq_new, con mascara de bins vacios."""
    sweep = sweeps[medium]
    n = n if n is not None else NMAX[(medium, mode)]
    grp = runs[(runs["medium"] == medium) & (runs["mode"] == mode)
               & (runs["n_total"] == n)].sort_values("rep")
    lstar = float(grp["lstar"].iloc[0])

    Cs, Is, qb = [], [], None
    for k in grp["key"]:
        q, c, i = raw_pair(sweep, k, lstar)
        qb, cb, ib = rebin_pair(q, c, i, dq_new)
        Cs.append(cb); Is.append(ib)
    Cs, Is = np.vstack(Cs), np.vstack(Is)

    # Un bin es utilizable si TODAS las replicas tienen senal ahi. Sin esto,
    # EPS convierte un bin vacio en eta = 1 exacto: el pico espureo de F1.
    ok = (Is > 0).all(axis=0) & (Cs > 0).all(axis=0)
    qb, per_rep = qb[ok], Cs[:, ok] / Is[:, ok]

    eta  = Cs[:, ok].sum(axis=0) / Is[:, ok].sum(axis=0)          # pooled
    tail = qb > 0.6 * qb[-1]
    eta  = eta - np.median(eta[tail]) + 1.0
    per_rep = per_rep - np.median(per_rep[:, tail], axis=1, keepdims=True) + 1.0
    half = T_CRIT * per_rep.std(axis=0, ddof=1) / np.sqrt(per_rep.shape[0])
    return qb, eta, half, int(ok.sum()), int((~ok).sum())


DQ_NEW = 0.20        # 10x mas grueso -> el ruido del analog baja ~3.2x

rows = []
for m in MEDIA:
    d = {}
    for mode in MODE_ORDER:
        q, eta, half, n_ok, n_bad = eta_point_rebin(m, mode, DQ_NEW)
        d[mode] = (q, eta, half)
        print(f"{m:12s} {mode:9s}: {n_ok} bins utiles, {n_bad} descartados")
    q     = d["estimator"][0]
    delta = d["estimator"][1] - d["analog"][1]
    hcomb = np.hypot(d["estimator"][2], d["analog"][2])
    sel   = q < Q_CONE
    z     = np.abs(delta[sel]) / np.where(hcomb[sel] > 0, hcomb[sel], np.nan)
    rows.append({"medium": m, "dq": DQ_NEW,
                 "rms_delta": np.sqrt(np.mean(delta[sel] ** 2)),
                 "max_z": np.nanmax(z), "frac_outside": np.nanmean(z > 1.0)})

print()
print(pd.DataFrame(rows).round(4).to_string(index=False))
# ojo: eta(0) y dq_FWHM NO se leen del bin crudo a esta resolucion.
# Pasar (q, eta, half) por el ajuste de cono de utils.cbs_fit y comparar
# ell_star y A con sus IC: esa es la comparacion que se cita.

Homogeneous  estimator: 200 bins utiles, 0 descartados
Homogeneous  analog   : 200 bins utiles, 0 descartados
Mixture      estimator: 200 bins utiles, 0 descartados
Mixture      analog   : 200 bins utiles, 0 descartados
Bilayer      estimator: 200 bins utiles, 0 descartados
Bilayer      analog   : 200 bins utiles, 0 descartados

     medium  dq  rms_delta  max_z  frac_outside
Homogeneous 0.2     0.0696 5.3629        0.7368
    Mixture 0.2     0.0226 3.9958        0.2632
    Bilayer 0.2     0.0325 2.1373        0.1842


In [12]:
def banda(ax, q, m, s, color, **kw):
    """Curva media con banda +-IC, espejada."""
    qs, ms, ss = mirror(q, m, s)
    ax.plot(qs, ms, color=color, **kw)
    ax.fill_between(qs, ms - ss, ms + ss, color=color, alpha=0.20, lw=0)


def leyenda_inferior(fig, ax, ncol=4):
    fig.legend(*ax.get_legend_handles_labels(), ncol=ncol,
               loc="outside lower center", frameon=False,
               columnspacing=1.4, handlelength=1.4)

# =========================================================================
# T -- transformacion (ancho, amplitud) entre modos + perfil corregido
#      Requiere rebin_pair() de la celda E.
# =========================================================================
from scipy.optimize import curve_fit

DQ_T    = 0.10      # resolucion de trabajo
Q_FIT_T = 6.0       # ventana de ajuste (el cono, sin la cola)


def rebinned(medium, mode, dq_new, n=None):
    """(q, eta_pooled, eta_por_replica, semiancho_IC) a resolucion dq_new."""
    sweep = sweeps[medium]
    n = n if n is not None else NMAX[(medium, mode)]
    grp = runs[(runs["medium"] == medium) & (runs["mode"] == mode)
               & (runs["n_total"] == n)].sort_values("rep")
    lstar = float(grp["lstar"].iloc[0])

    Cs, Is, qb = [], [], None
    for k in grp["key"]:
        q, c, i = raw_pair(sweep, k, lstar)
        qb, cb, ib = rebin_pair(q, c, i, dq_new)
        Cs.append(cb); Is.append(ib)
    Cs, Is = np.vstack(Cs), np.vstack(Is)

    ok   = (Is > 0).all(axis=0) & (Cs > 0).all(axis=0)
    qb   = qb[ok]
    tail = qb > 0.6 * qb[-1]

    per_rep = Cs[:, ok] / Is[:, ok]
    per_rep = per_rep - np.median(per_rep[:, tail], axis=1, keepdims=True) + 1.0
    eta = Cs[:, ok].sum(axis=0) / Is[:, ok].sum(axis=0)
    eta = eta - np.median(eta[tail]) + 1.0
    half = T_CRIT * per_rep.std(axis=0, ddof=1) / np.sqrt(per_rep.shape[0])
    return qb, eta, per_rep, half


def _model(qq, s, A, q_ref, e_ref):
    """1 + A*[eta_est(q/s) - 1], por interpolacion sobre la referencia."""
    return 1.0 + A * np.interp(qq / s, q_ref, e_ref - 1.0,
                               left=e_ref[0] - 1.0, right=0.0)


def fit_sA(q, y, q_ref, e_ref, q_fit=Q_FIT_T):
    m = (q > 0) & (q < q_fit) & np.isfinite(y)
    popt, _ = curve_fit(lambda qq, s, A: _model(qq, s, A, q_ref, e_ref),
                        q[m], y[m], p0=[1.0, 1.0],
                        bounds=([0.5, 0.5], [2.0, 1.5]), maxfev=20000)
    return popt


def apply_transform(q, eta, s, A):
    """Lleva el analog al marco del estimador."""
    return 1.0 + np.interp(s * q, q, eta - 1.0,
                           left=eta[0] - 1.0, right=0.0) / A


TRANS, rows = {}, []
for m in MEDIA:
    q_ref, e_ref, _,     h_ref = rebinned(m, "estimator", DQ_T)
    q_a,   e_a,  rep_a,  h_a   = rebinned(m, "analog",    DQ_T)

    v = np.array([fit_sA(q_a, r, q_ref, e_ref) for r in rep_a])   # (R, 2)
    (s_hat, A_hat) = v.mean(axis=0)
    ci = T_CRIT * v.std(axis=0, ddof=1) / np.sqrt(v.shape[0])
    TRANS[m] = dict(s=s_hat, A=A_hat, ci_s=ci[0], ci_A=ci[1],
                    q_ref=q_ref, e_ref=e_ref, h_ref=h_ref,
                    q_a=q_a, e_a=e_a, h_a=h_a)

    e_corr = apply_transform(q_a, e_a, s_hat, A_hat)
    sel    = q_a < Q_FIT_T
    rows.append({
        "medium":            m,
        "s (ancho ana/est)": s_hat,     "CI_s": ci[0],
        "A (amp ana/est)":   A_hat,     "CI_A": ci[1],
        "lstar_ana/lstar_est": 1.0 / s_hat,
        "rms antes":  np.sqrt(np.mean((e_a[sel]     - np.interp(q_a[sel], q_ref, e_ref)) ** 2)),
        "rms despues": np.sqrt(np.mean((e_corr[sel] - np.interp(q_a[sel], q_ref, e_ref)) ** 2)),
    })

df_T = pd.DataFrame(rows)
df_T.to_csv(FIGDIR / "conv_mode_transform.csv", index=False)
print(df_T.round(4).to_string(index=False))

     medium  s (ancho ana/est)   CI_s  A (amp ana/est)   CI_A  lstar_ana/lstar_est  rms antes  rms despues
Homogeneous             0.5749 0.0566           1.0912 0.0790               1.7394     0.0784       0.0175
    Mixture             0.9054 0.0244           0.9624 0.0373               1.1045     0.0262       0.0081
    Bilayer             0.7513 0.2015           1.0582 0.1654               1.3310     0.0367       0.0204


## 4b. Réplicas renormalizadas y grilla común

Dos utilidades que necesita todo lo que sigue.

`en_grilla` lleva cualquier perfil a la grilla de otro. Hace falta porque el $q$
de salida del re-binning es el **centroide pesado del bin**, y ese centroide
depende de la ocupación: dos modos con la misma `EDGES` no tienen exactamente el
mismo vector $q$. Comparar bin a bin sin interpolar mete un desplazamiento
angular espurio en el residuo.

`eta_rep_ren` son las cinco réplicas analógicas ya renormalizadas. La §4 sólo
guardaba el pooled y la banda; para los $z$ por bin y para los ajustes con piso
hacen falta las réplicas individuales en el marco del estimador.

In [13]:
def en_grilla(q_dst, q_src, y_src):
    """Interpola y_src(q_src) sobre q_dst. Fuera del soporte -> nan (no extrapola)."""
    q_dst = np.asarray(q_dst, float)
    q_src = np.asarray(q_src, float)
    y_src = np.asarray(y_src, float)
    ok = np.isfinite(q_src) & np.isfinite(y_src)
    out = np.full(q_dst.shape, np.nan)
    if ok.sum() < 2:
        return out
    val = np.isfinite(q_dst)
    out[val] = np.interp(q_dst[val], q_src[ok], y_src[ok], left=np.nan, right=np.nan)
    return out


# Replicas analogicas en el marco del estimador. El estimador no se toca.
_rep_ren = []
for _, r in perfiles.iterrows():
    if r["mode"] != "analog":
        _rep_ren.append(r["eta_rep"])
        continue
    s, A = renorm.loc[r["medium"], "s"], renorm.loc[r["medium"], "A"]
    _rep_ren.append(np.vstack([renormaliza(r["q"], e, s, A) for e in r["eta_rep"]]))

perfiles["eta_rep_ren"] = pd.Series(_rep_ren, index=perfiles.index, dtype=object)


def par_convergido(medio):
    """Las dos corridas a N_max de un medio, sobre la grilla del estimador.

    Devuelve (q, eta_est, SE_est, eta_ana_ren, SE_ana_ren). SE = err / t, es
    decir la desviacion estandar de la media sobre replicas, sin el factor t:
    asi los z de mas abajo se leen contra el umbral t_{0.975,4} explicito.
    """
    r_e, r_a = fila(medio, "estimator"), fila(medio, "analog")
    q = r_e["q"]
    return (q,
            r_e["eta_ren"], r_e["err_ren"] / T_CRIT,
            en_grilla(q, r_a["q"], r_a["eta_ren"]),
            en_grilla(q, r_a["q"], r_a["err_ren"]) / T_CRIT)

NameError: name 'perfiles' is not defined

## 5b. ¿Ruido o sesgo? La discrepancia contra su propia barra de error

El residuo crudo $D_{\rm rms}$ no dice nada por sí solo: $0.045$ en $\eta$ no es
grande ni pequeño hasta compararlo contra **cuánto se esperaría por ruido**. Si
los dos modos midieran lo mismo y sólo difirieran por estadística, el residuo
punto a punto tendría desviación

$$\sigma_\Delta(q) \;=\; \sqrt{\sigma^2_{\rm est}(q) + \sigma^2_{\rm ana}(q)}$$

con $\sigma = s/\sqrt{R}$ de cada modo. Entonces

$$\text{razón} \;=\; \frac{D_{\rm rms}}{\bigl\langle \sigma_\Delta^2 \bigr\rangle^{1/2}}
\;\xrightarrow[\text{sin sesgo}]{}\; 1 ,$$

y el exceso da una **cota superior al sesgo** residual entre modos,

$$b \;=\; \sqrt{\max\bigl(D_{\rm rms}^2 - \langle\sigma_\Delta^2\rangle,\; 0\bigr)} .$$

Esa cota, citada en unidades de la amplitud del cono, es el número que responde
la pregunta del apéndice: *cuánto puede valer, como máximo, la diferencia
sistemática entre estimador y analógico renormalizado*.

La fracción de puntos fuera de la banda se compara contra el $\sim 5\%$ nominal.
Es aproximado: los bins vecinos están correlacionados en $q$, así que el número
efectivo de grados de libertad es menor que el de bins.

In [ ]:
filas = []
for medio in MEDIOS:
    q, eta_e, se_e, eta_a, se_a = par_convergido(medio)

    delta = eta_e - eta_a                      # estimador - analogico RENORMALIZADO
    se_d  = np.hypot(se_e, se_a)               # ruido esperado de esa diferencia
    ok = (q > 0) & (q < Q_CONO) & np.isfinite(delta) & np.isfinite(se_d) & (se_d > 0)

    D = float(np.sqrt(np.mean(delta[ok] ** 2)))        # discrepancia medida
    S = float(np.sqrt(np.mean(se_d[ok] ** 2)))         # discrepancia esperada por ruido
    z = np.abs(delta[ok]) / se_d[ok]
    amp = float(np.mean(eta_e[ok] - 1.0))              # amplitud media del cono en la ventana

    filas.append(dict(medium=medio,
                      D_rms=D, sigma_esperada=S, razon=D / S,
                      sesgo_cota=np.sqrt(max(D ** 2 - S ** 2, 0.0)),
                      sesgo_rel=np.sqrt(max(D ** 2 - S ** 2, 0.0)) / amp,
                      z_mediano=float(np.median(z)),
                      frac_fuera=float(np.mean(z > T_CRIT)),
                      n_bins=int(ok.sum())))

consistencia = pd.DataFrame(filas).set_index("medium")
consistencia.to_csv(FIGDIR / "tabla_consistencia.csv")

print("razon -> 1        : la diferencia entre modos es el ruido de las referencias")
print("sesgo_cota        : cota superior al sesgo sistematico, en unidades de eta")
print("sesgo_rel         : la misma cota, como fraccion de la amplitud del cono")
print(f"frac_fuera        : esperada ~0.05 bajo H0 (banda t_{{0.975,4}}); bins correlacionados en q")
consistencia.round(4)

### Fig. 1c — el residuo en unidades de su propia incertidumbre

Es la Fig. 1b dividida punto a punto por $\sigma_\Delta$. La banda gris es el
umbral $\pm t_{0.975,4}$: si la curva vive dentro salvo excursiones aisladas, la
discrepancia es ruido. Un sesgo se vería como una excursión **de un solo signo**
sostenida sobre un tramo de $q$, no como oscilación.

In [ ]:
fig, axes = plt.subplots(1, len(MEDIOS), squeeze=False, sharey=True,
                         figsize=(TEXTWIDTH_IN, 0.34 * TEXTWIDTH_IN))
axes, tags = axes[0], iter("abc")

for ax, medio in zip(axes, MEDIOS):
    q, eta_e, se_e, eta_a, se_a = par_convergido(medio)
    se_d = np.hypot(se_e, se_a)
    z = (eta_e - eta_a) / np.where(se_d > 0, se_d, np.nan)

    ax.plot(*espejo(q, z), "k-", lw=0.7)
    ax.axhspan(-T_CRIT, T_CRIT, color="gray", alpha=0.25, lw=0)
    ax.axhline(0.0, ls="--", c="gray", lw=0.7)
    ax.set_xlim(*XLIM)
    ax.set_ylim(-6, 6)
    ax.set_xlabel(r"$q$")
    ax.set_title(rf"({next(tags)}) {medio}", loc="left")
    ax.grid(False)

axes[0].set_ylabel(r"$z = \Delta\eta\,/\,\sigma_\Delta$")
fig.savefig(FIGDIR / "f1c_zscore.pdf")
plt.show()

## 6b. $D_{\rm rms}$ contra el **otro** modo

La §6 mide la distancia de cada corrida a la referencia de su propio modo. Eso
prueba que cada modo converge a *algo*, no que converjan **al mismo algo** — que
es justamente lo que pide el apéndice.

Aquí se calculan las dos cosas, para cada punto de las dos escaleras:

- `D_own`: contra la referencia del propio modo (lo que ya había).
- `D_cross`: contra la referencia del **otro** modo, siempre con el analógico
  renormalizado. Aquí sí entra el último punto de cada escalera, porque la
  referencia es externa y la distancia no es cero por construcción.

`D_cross` no baja a cero: satura en el ruido de la referencia externa, que se
calcula aparte y se guarda en `piso`. Si las dos curvas `D_cross` bajan hasta ese
piso y no por debajo, los dos modos convergen al mismo perfil y la comparación
está limitada por estadística, no por sesgo.

In [ ]:
def d_rms_par(q_ref, y_ref, q, y):
    """RMS de (y_ref - y) en 0 < q < Q_CONO, con y llevado a la grilla de referencia."""
    y_i = en_grilla(q_ref, q, y)
    ok = (q_ref > 0) & (q_ref < Q_CONO) & np.isfinite(y_i) & np.isfinite(y_ref)
    return float(np.sqrt(np.mean((y_ref[ok] - y_i[ok]) ** 2))) if ok.any() else np.nan


OTRO = {"estimator": "analog", "analog": "estimator"}

filas = []
for medio in MEDIOS:
    ref = {md: fila(medio, md) for md in MODOS}
    for modo in MODOS:
        r_prop, r_otro = ref[modo], ref[OTRO[modo]]
        for N in NS[(medio, modo)]:
            r = fila(medio, modo, N)
            filas.append(dict(
                medium=medio, mode=modo, N=N,
                D_own=(np.nan if N == r_prop["N"] else
                       d_rms_par(r_prop["q"], r_prop["eta_ren"], r["q"], r["eta_ren"])),
                D_cross=d_rms_par(r_otro["q"], r_otro["eta_ren"], r["q"], r["eta_ren"]),
            ))

conv = pd.DataFrame(filas).sort_values(["medium", "mode", "N"]).reset_index(drop=True)
conv.to_csv(FIGDIR / "tabla_convergencia.csv", index=False)

# Piso esperado de cada serie cruzada: el ruido de la referencia CONTRA la que se
# compara. Para la escalera del estimador, esa referencia es el analogico
# renormalizado a N_max, y viceversa.
piso = {}
for medio in MEDIOS:
    q, eta_e, se_e, eta_a, se_a = par_convergido(medio)
    ok = (q > 0) & (q < Q_CONO)
    piso[(medio, "estimator")] = float(np.sqrt(np.nanmean(se_a[ok] ** 2)))
    piso[(medio, "analog")]    = float(np.sqrt(np.nanmean(se_e[ok] ** 2)))

print("piso esperado de D_cross (ruido de la referencia externa):")
for k, v in piso.items():
    print(f"    {k[0]:12s} serie {k[1]:9s} -> {v:.4f}")

## 6c. Ajuste con piso: exponente fijo en $-1/2$

El ajuste de ley de potencia libre da exponentes entre $-0.33$ y $-0.47$, que no
confirman $N^{-1/2}$: lo aproximan. Y el motivo no es que el Monte Carlo falle,
es que **medir distancia contra una referencia ruidosa satura la métrica** y
aplana el exponente aparente. El modelo correcto separa las dos cosas:

$$D_{\rm rms}^2(N) \;=\; \frac{a}{N} \;+\; c^2 ,$$

con el exponente **fijo** en el valor Monte Carlo y $c$ absorbiendo el ruido de
la referencia. Se ajusta como una recta de $D^2$ contra $1/N$ — dos parámetros,
ninguno libre en el exponente. La prueba es entonces si el $c$ ajustado coincide
con el `piso` predicho en 6b: si coincide, la saturación es exactamente el ruido
de la referencia y no queda nada sistemático por explicar.

El exponente libre se conserva en la tabla sólo como diagnóstico, no como
resultado citable.

In [ ]:
def ajuste_con_piso(ns, D):
    """D^2 = a/N + c^2, exponente fijo. Recta de D^2 contra 1/N."""
    x = 1.0 / np.asarray(ns, float)
    y = np.asarray(D, float) ** 2
    a, c2 = np.polyfit(x, y, 1)
    return float(a), float(np.sqrt(max(c2, 0.0)))


def exponente_libre(ns, D):
    return float(np.polyfit(np.log(np.asarray(ns, float)), np.log(np.asarray(D, float)), 1)[0])


filas = []
for medio in MEDIOS:
    for modo in MODOS:
        sub = conv[(conv["medium"] == medio) & (conv["mode"] == modo)]
        for etiqueta, col in (("own", "D_own"), ("cross", "D_cross")):
            d = sub.dropna(subset=[col]).sort_values("N")
            if len(d) < 3:
                continue
            a, c = ajuste_con_piso(d["N"].to_numpy(), d[col].to_numpy())
            filas.append(dict(medium=medio, mode=modo, referencia=etiqueta,
                              n_puntos=len(d),
                              a=a, c_ajustado=c,
                              c_esperado=piso[(medio, modo)] if etiqueta == "cross" else np.nan,
                              exp_libre=exponente_libre(d["N"].to_numpy(), d[col].to_numpy())))

ajustes = pd.DataFrame(filas).set_index(["medium", "mode", "referencia"])
ajustes["c_aj/c_esp"] = ajustes["c_ajustado"] / ajustes["c_esperado"]
ajustes.to_csv(FIGDIR / "tabla_ajuste_convergencia.csv")

print("c_aj/c_esp ~ 1  ->  la saturacion de D_cross es el ruido de la referencia, nada mas")
print("exp_libre       ->  diagnostico; sesgado hacia arriba por el piso, no citar como confirmacion")
ajustes.round(4)

### Fig. 3b — convergencia entre modos

Marcadores llenos: cada escalera contra la referencia del **otro** modo
(analógico siempre renormalizado). Marcadores huecos: contra la referencia propia
— la métrica de la §6, que se incluye para que se vea que baja más, como debe,
porque no arrastra el ruido de una referencia externa.

Línea continua: el ajuste $D^2 = a/N + c^2$. Línea punteada: el piso predicho de
forma independiente a partir de las barras de error de la referencia. Que la
curva ajustada se acueste sobre la línea punteada es el resultado.

In [ ]:
fig, axes = plt.subplots(1, len(MEDIOS), squeeze=False, sharey=True,
                         figsize=(TEXTWIDTH_IN, 0.38 * TEXTWIDTH_IN))
axes, tags = axes[0], iter("abc")

for ax, medio in zip(axes, MEDIOS):
    for modo in MODOS:
        c_modo = ESTILO[modo]["color"]
        sub = conv[(conv["medium"] == medio) & (conv["mode"] == modo)]

        d = sub.dropna(subset=["D_cross"]).sort_values("N")
        ax.plot(d["N"], d["D_cross"], ls="none", marker=MARCA[modo], ms=3.5, color=c_modo)

        o = sub.dropna(subset=["D_own"]).sort_values("N")
        ax.plot(o["N"], o["D_own"], ls="none", marker=MARCA[modo], ms=3.5,
                mfc="none", color=c_modo, alpha=0.55)

        if (medio, modo, "cross") in ajustes.index:
            f = ajustes.loc[(medio, modo, "cross")]
            nn = np.logspace(np.log10(d["N"].min()), np.log10(d["N"].max()), 200)
            ax.plot(nn, np.sqrt(f["a"] / nn + f["c_ajustado"] ** 2),
                    "-", lw=0.9, color=c_modo, zorder=0)

        ax.axhline(piso[(medio, modo)], ls=":", lw=0.8, color=c_modo, zorder=0)

    ax.set(xscale="log", yscale="log", xlabel=r"$N$")
    ax.set_title(rf"({next(tags)}) {medio}", loc="left")
    ax.grid(False)

axes[0].set_ylabel(r"$D_{\mathrm{rms}}(\eta)$")

handles = [Line2D([], [], ls="none", marker=MARCA[md], color=ESTILO[md]["color"],
                  label=f"{md} vs. other mode") for md in MODOS]
handles += [Line2D([], [], ls="none", marker="o", mfc="none", color="k",
                   label="vs. own reference"),
            Line2D([], [], ls=":", color="k", label="reference noise floor")]
fig.legend(handles=handles, ncol=2, loc="outside lower center", frameon=False,
           handlelength=1.6, columnspacing=1.2)
fig.savefig(FIGDIR / "f3b_convergencia_cruzada.pdf")
plt.show()

## 7b. Speedup a igual precisión

Decir *"tres órdenes de magnitud menos historias"* compara $5\times10^5$ contra
$10^9$, pero el estimador ya está convergido muy por debajo de $5\times10^5$: esa
cifra subestima la ganancia y además no es una medida de nada, porque los dos
puntos no tienen la misma precisión.

La comparación operativa es la inversa: **qué necesita el estimador para igualar
la $\sigma[\eta(0)]$ que el analógico alcanza en $N_{\max}$**. Con
$\sigma^2 \propto 1/N$ y $T \propto N$,

$$N_{\rm eq} = N_{\rm est}\left(\frac{\sigma_{\rm est}}{\sigma_{\rm ana}}\right)^2 ,
\qquad
T_{\rm eq} = \frac{1}{{\rm FOM}_{\rm est}\,\sigma_{\rm ana}^2} .$$

Las dos ganancias, en historias y en tiempo de CPU, son las que van al texto. La
$\sigma$ del analógico es la del ápice **ya renormalizado** (dividido por $A$),
que es como se cita en toda la sección.

In [ ]:
filas = []
for medio in MEDIOS:
    c = costo[costo["medium"] == medio]
    e = c[(c["mode"] == "estimator") & (c["N"] == NMAX[(medio, "estimator")])].iloc[0]
    a = c[(c["mode"] == "analog")    & (c["N"] == NMAX[(medio, "analog")])].iloc[0]

    N_eq = float(e["N"] * (e["sigma_eta0"] / a["sigma_eta0"]) ** 2)
    T_eq = float(1.0 / (e["FOM"] * a["sigma_eta0"] ** 2))

    filas.append(dict(medium=medio,
                      sigma_objetivo=a["sigma_eta0"],
                      N_analog=a["N"], T_analog_h=a["T_wall"] / 3600,
                      N_est_equiv=N_eq, T_est_equiv_h=T_eq / 3600,
                      ganancia_N=a["N"] / N_eq,
                      ganancia_T=a["T_wall"] / T_eq))

igual = pd.DataFrame(filas).set_index("medium")
igual.to_csv(FIGDIR / "tabla_igual_precision.csv")

print("Cuanto necesita el estimador para IGUALAR la sigma[eta(0)] del analogico en N_max.")
print("T_est_equiv extrapola con T ~ N; por debajo de N ~ 1e5 el setup fijo domina")
print("y la extrapolacion es optimista: citarla como cota, no como medida.")
igual.round(4)

## 7c. De dónde sale el costo

Dos comprobaciones que hay que poder responder si preguntan por la tabla de
tiempos:

1. **¿Son comparables?** Los wall times sólo se pueden poner en la misma tabla si
   las tres campañas comparten número de hilos y tamaño del detector angular. La
   primera celda vuelca los parámetros relevantes tal como quedaron archivados en
   el HDF5, sin suponer nombres.
2. **¿Por qué la mezcla cuesta más?** El costo por historia lo dice directo. Si
   la mezcla sale varias veces más cara que el homogéneo al mismo $N$, el
   candidato es el número de eventos por historia (la mezcla tiene $\mu_s$ y $g$
   propios) más el sorteo de especie en cada vértice — no el detector, que es el
   mismo. Ese número es el que hay que citar en el texto en lugar de dejar la
   anomalía sin explicar.

In [ ]:
CLAVES = ("thread", "n_theta", "n_phi", "theta", "phi", "q_max", "bin",
          "photons", "runtime", "mode", "lstar", "mu_s", "g_")

print("=== parametros archivados por campana y modo ===")
for medio in MEDIOS:
    for modo in MODOS:
        key = runs[(runs["medium"] == medio) & (runs["mode"] == modo)]["key"].iloc[0]
        p = sweeps[medio][key].params_flat
        rel = {k: v for k, v in p.items() if any(c in k.lower() for c in CLAVES)}
        print(f"\n--- {medio} / {modo} ---")
        for k in sorted(rel):
            print(f"    {k:40s} {rel[k]}")

In [ ]:
# Costo por historia en N_max: T_wall y N son ambos sumas sobre las 5 replicas,
# asi que el cociente es directamente el costo medio de una historia.
sel = costo.apply(lambda r: r["N"] == NMAX[(r["medium"], r["mode"])], axis=1)
por_historia = (costo[sel]
                .assign(us_por_historia=lambda d: 1e6 * d["T_wall"] / d["N"])
                .pivot(index="medium", columns="mode", values="us_por_historia")
                .loc[MEDIOS])
por_historia["razon_est_sobre_ana"] = por_historia["estimator"] / por_historia["analog"]
por_historia["vs_homogeneo_est"] = (por_historia["estimator"] /
                                    por_historia["estimator"].iloc[0])
por_historia.to_csv(FIGDIR / "tabla_costo_por_historia.csv")

print("us por historia en N_max. razon_est_sobre_ana = sobrecosto del next-event")
print("por historia; vs_homogeneo_est = cuanto mas cara es cada historia respecto")
print("al medio homogeneo, con el MISMO detector.")
por_historia.round(3)

## 9b. Qué números van al `.tex`

Después de correr todo, el apéndice se reescribe con estos, no con los RMS
sueltos:

| dónde | número | de dónde sale |
|---|---|---|
| §A.1 | razón $D_{\rm rms}/\sigma_\Delta$ y cota de sesgo $b$ | `tabla_consistencia.csv` |
| §A.1 | fracción fuera de banda vs. el 5% nominal | `tabla_consistencia.csv` |
| §A.2 | piso ajustado $c$ vs. piso predicho | `tabla_ajuste_convergencia.csv` |
| §A.2 | coeficiente $a$ del término $N^{-1}$ | `tabla_ajuste_convergencia.csv` |
| §A.3 | $N_{\rm eq}$, $T_{\rm eq}$ y las dos ganancias | `tabla_igual_precision.csv` |
| §A.3 | costo por historia y sobrecosto de la mezcla | `tabla_costo_por_historia.csv` |

Figuras nuevas: `f1c_zscore.pdf` (§A.1) y `f3b_convergencia_cruzada.pdf` (§A.2,
en lugar de la actual Fig. A.3 o junto a ella).

**Alcance.** Todo esto es el canal circular de helicidad conservada, donde el
single backscatter no entra al detector. Los canales donde sí entra necesitan una
verificación separada del depósito de orden 1 entre los dos modos; conviene
decirlo explícito al final de §A.1 en vez de dejar la pregunta abierta.